# Two-stage probabilistic forecasting

This notebook covers both supported scenarios: **Negative Binomial** for non-negative integer counts and **Gamma** for strictly positive continuous values. Each example shows the PPF, CDF, the applicable probability function, and Newsvendor optimization.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression
import sys
import os
sys.path.append(os.path.abspath("../.."))
from tinyshift.forecasting import (
    FirstStageForecasterEvaluator,
    GammaFamily,
    NewsvendorOptimizer,
    TwoStageForecasterEvaluator,
    TwoStageForecasterWrapper,
)

pd.set_option("display.max_columns", 20)

## 1. Discrete scenario: Negative Binomial

The default family models count data and supports `cdf`, `ppf`, and `pmf`. Its quantiles and optimized quantities are integers.

In [2]:
def make_count_data(n_periods=180, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["store_A", "store_B"]):
        mean = 2.5 + offset + np.linspace(0, 1, n_periods) + 1.2 * (dates.dayofweek >= 5)
        size = 4.0
        y = rng.negative_binomial(size, size / (size + mean))
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

count_data = make_count_data()
count_data.head()

,unique_id,ds,y
0,store_A,2024-01-01,2
1,store_A,2024-01-02,1
2,store_A,2024-01-03,3
3,store_A,2024-01-04,0
4,store_A,2024-01-05,6


In [3]:
count_fcst = MLForecast(
    models=[LinearRegression()], freq="D", lags=[1, 7, 14], date_features=["dayofweek"]
)
count_model = TwoStageForecasterWrapper(count_fcst).fit(
    count_data, h=7, n_windows=4
)

### Discrete PPF, CDF, and PMF

`predict_distribution` returns one self-contained, panel-aligned forecast. Use `to_frame()` for point forecasts and call `ppf`, `cdf`, `pmf`, or `interval` directly; each method returns a DataFrame on the same panel grid. A scalar is evaluated for every forecast row, while a one-dimensional input defines a common grid.

In [4]:
count_forecast = count_model.predict_distribution(h=7)
count_frame = count_forecast.to_frame()
count_distribution = count_forecast.distribution  # used by decision utilities below
count_forecast.ppf([0.50, 0.90, 0.95]).head()

,unique_id,ds,lambda_t,r_dispersion,lambda_t-q-50,lambda_t-q-90,lambda_t-q-95
0,store_A,2024-06-29,4.175291,4.218636,4,8,10
1,store_A,2024-06-30,4.396524,4.218636,4,8,10
2,store_A,2024-07-01,3.488963,4.218636,3,7,8
3,store_A,2024-07-02,3.806164,4.218636,3,7,9
4,store_A,2024-07-03,3.846992,4.218636,3,7,9


In [5]:
# Exact probabilities P(Y=k), plus the probability above max_k.
max_k = 8
count_units = np.arange(max_k + 1)
count_pmf = count_distribution.pmf(count_units)
count_probabilities = count_frame.copy()
for index, unit in enumerate(count_units):
    count_probabilities[f"P(Y={unit})"] = count_pmf[:, index]
count_probabilities[f"P(Y>{max_k})"] = 1.0 - count_pmf.sum(axis=1)
count_probabilities.head()

,unique_id,ds,lambda_t,r_dispersion,P(Y=0),P(Y=1),P(Y=2),P(Y=3),P(Y=4),P(Y=5),P(Y=6),P(Y=7),P(Y=8),P(Y>8)
0,store_A,2024-06-29,4.175291,4.218636,0.054891,0.115185,0.149501,0.154148,0.138374,0.113137,0.086466,0.062786,0.043796,0.081718
1,store_A,2024-06-30,4.396524,4.218636,0.049186,0.105890,0.141003,0.149159,0.137369,0.115230,0.090350,0.067308,0.048169,0.096336
2,store_A,2024-07-01,3.488963,4.218636,0.078666,0.150222,0.177434,0.166490,0.136007,0.101197,0.070382,0.046509,0.029523,0.043570
3,store_A,2024-07-02,3.806164,4.218636,0.066358,0.132776,0.164324,0.161558,0.138285,0.107810,0.078565,0.054397,0.036181,0.059747
4,store_A,2024-07-03,3.846992,4.218636,0.064953,0.130693,0.162653,0.160812,0.138420,0.108520,0.079526,0.055372,0.037036,0.062017


### Optimize discrete inventory

`NewsvendorOptimizer.optimize` computes the Newsvendor critical ratio `underage_cost / (underage_cost + overage_cost)` and evaluates the distribution PPF at that probability. Costs can be scalars, `cost_df` columns, or dictionaries.

In [6]:
discrete_plan = NewsvendorOptimizer.optimize(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
)
discrete_plan.head()

,unique_id,ds,lambda_t,r_dispersion,critical_ratio,y_optimal
0,store_A,2024-06-29,4.175291,4.218636,0.833333,7
1,store_A,2024-06-30,4.396524,4.218636,0.833333,7
2,store_A,2024-07-01,3.488963,4.218636,0.833333,6
3,store_A,2024-07-02,3.806164,4.218636,0.833333,6
4,store_A,2024-07-03,3.846992,4.218636,0.833333,6


## 2. Continuous scenario: Gamma

Pass `GammaFamily()` for continuous, strictly positive targets. Gamma supports `cdf` and `ppf`; 

In [7]:
def make_continuous_data(n_periods=180, seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-01-01", periods=n_periods, freq="D")
    frames = []
    for offset, unique_id in enumerate(["region_A", "region_B"]):
        mean = 8 + 2 * offset + np.linspace(0, 2, n_periods) + 0.8 * np.sin(2 * np.pi * np.arange(n_periods) / 7)
        shape = 12.0
        y = rng.gamma(shape=shape, scale=mean / shape)
        frames.append(pd.DataFrame({"unique_id": unique_id, "ds": dates, "y": y}))
    return pd.concat(frames, ignore_index=True)

continuous_data = make_continuous_data()
continuous_data.head()

,unique_id,ds,y
0,region_A,2024-01-01,7.780579
1,region_A,2024-01-02,7.740684
2,region_A,2024-01-03,7.468419
3,region_A,2024-01-04,8.292144
4,region_A,2024-01-05,6.456301


In [8]:
continuous_fcst = MLForecast(
    models=[LinearRegression()], freq="D", lags=[1, 7, 14], date_features=["dayofweek"]
)
continuous_model = TwoStageForecasterWrapper(
    continuous_fcst, distribution=GammaFamily()
).fit(continuous_data, h=7, n_windows=4)

### Continuous PPF and CDF

Gamma quantiles are real-valued. For interval probabilities, subtract two CDF evaluations—for example, `CDF(12) - CDF(8)` gives the probability that demand is between 8 and 12.

In [9]:
continuous_forecast = continuous_model.predict_distribution(h=7)
continuous_frame = continuous_forecast.to_frame()
continuous_distribution = continuous_forecast.distribution
continuous_forecast.ppf([0.05, 0.50, 0.95]).head()

,unique_id,ds,lambda_t,shape_dispersion,lambda_t-q-5,lambda_t-q-50,lambda_t-q-95
0,region_A,2024-06-29,9.811050,12.249481,5.698063,9.545401,14.830567
1,region_A,2024-06-30,8.415111,12.486761,4.916651,8.191567,12.676411
2,region_A,2024-07-01,9.425459,14.548517,5.760905,9.210406,13.823837
3,region_A,2024-07-02,10.622726,12.503618,6.209075,10.340917,15.998047
4,region_A,2024-07-03,10.414562,13.235718,6.193584,10.153484,15.526444


### Optimize continuous inventory

The workflow is the same, but the Gamma PPF returns a continuous optimal quantity. Row-level decision costs are passed separately through `cost_df`; they are not forecast features.

In [10]:
future_costs = continuous_fcst.make_future_dataframe(h=7)
future_costs["shortage_cost"] = np.where(
    future_costs["unique_id"] == "region_A", 6.0, 9.0
)
future_costs["holding_cost"] = 2.0

continuous_plan = NewsvendorOptimizer.optimize(
    continuous_frame,
    continuous_distribution,
    underage_cost="shortage_cost",
    overage_cost="holding_cost",
    cost_df=future_costs,
)
continuous_plan.head()

,unique_id,ds,lambda_t,shape_dispersion,critical_ratio,y_optimal
0,region_A,2024-06-29,9.811050,12.249481,0.75,11.529119
1,region_A,2024-06-30,8.415111,12.486761,0.75,9.876272
2,region_A,2024-07-01,9.425459,14.548517,0.75,10.954287
3,region_A,2024-07-02,10.622726,12.503618,0.75,12.466106
4,region_A,2024-07-03,10.414562,13.235718,0.75,12.176732


### Marginal benefit by inventory unit

For discrete distributions, `NewsvendorOptimizer.marginal_benefit` evaluates the expected net benefit of adding each inventory unit. Positive values favor stocking the unit; negative values indicate that its expected overage cost is larger than its shortage benefit. Use `max_k` to evaluate every unit from zero through an upper bound.

In [11]:
marginal_benefits = NewsvendorOptimizer.marginal_benefit(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
    max_k=8,
)
marginal_benefits.head()

,unique_id,ds,lambda_t,r_dispersion,MB(k=0),MB(k=1),MB(k=2),MB(k=3),MB(k=4),MB(k=5),MB(k=6),MB(k=7),MB(k=8)
0,store_A,2024-06-29,4.175291,4.218636,10.0,9.341309,7.959095,6.165088,4.315311,2.654822,1.297173,0.259586,-0.493840
1,store_A,2024-06-30,4.396524,4.218636,10.0,9.409773,8.139091,6.447052,4.657147,3.008713,1.625952,0.541753,-0.265946
2,store_A,2024-07-01,3.488963,4.218636,10.0,9.056014,7.253349,5.124135,3.126251,1.494166,0.279799,-0.564784,-1.122887
3,store_A,2024-07-02,3.806164,4.218636,10.0,9.203701,7.610388,5.638504,3.699812,2.040390,0.746670,-0.196106,-0.848871
4,store_A,2024-07-03,3.846992,4.218636,10.0,9.220568,7.652253,5.700417,3.770671,2.109637,0.807395,-0.146915,-0.811374


Use `units` instead when only specific inventory levels are relevant. The supplied order is preserved, and `max_k` and `units` are mutually exclusive. A stepped `range` provides a middle ground between a dense interval and a manually selected sparse grid.

In [12]:
sparse_marginal_benefits = NewsvendorOptimizer.marginal_benefit(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
    units=[2, 5, 8],
)
sparse_marginal_benefits.head()

,unique_id,ds,lambda_t,r_dispersion,MB(k=2),MB(k=5),MB(k=8)
0,store_A,2024-06-29,4.175291,4.218636,7.959095,2.654822,-0.493840
1,store_A,2024-06-30,4.396524,4.218636,8.139091,3.008713,-0.265946
2,store_A,2024-07-01,3.488963,4.218636,7.253349,1.494166,-1.122887
3,store_A,2024-07-02,3.806164,4.218636,7.610388,2.040390,-0.848871
4,store_A,2024-07-03,3.846992,4.218636,7.652253,2.109637,-0.811374


In [13]:
stepped_marginal_benefits = NewsvendorOptimizer.marginal_benefit(
    count_frame,
    count_distribution,
    underage_cost=10.0,
    overage_cost=2.0,
    units=range(0, 21, 5),
)
stepped_marginal_benefits.head()

,unique_id,ds,lambda_t,r_dispersion,MB(k=0),MB(k=5),MB(k=10),MB(k=15),MB(k=20)
0,store_A,2024-06-29,4.175291,4.218636,10.0,2.654822,-1.374295,-1.947017,-1.996489
1,store_A,2024-06-30,4.396524,4.218636,10.0,3.008713,-1.244443,-1.927649,-1.994564
2,store_A,2024-07-01,3.488963,4.218636,10.0,1.494166,-1.694884,-1.983618,-1.999317
3,store_A,2024-07-02,3.806164,4.218636,10.0,2.040390,-1.562614,-1.970557,-1.998455
4,store_A,2024-07-03,3.846992,4.218636,10.0,2.109637,-1.543584,-1.968434,-1.998297


## Out-of-sample evaluation and calibration

Forecast diagnostics should use observations that were not used for fitting. The following example reserves the final 14 days, fits a separate model on the earlier history, and joins predictions to the held-out targets.

In [14]:
evaluation_horizon = 14
cutoff = count_data["ds"].max() - pd.Timedelta(days=evaluation_horizon)
evaluation_train = count_data[count_data["ds"] <= cutoff].copy()
evaluation_test = count_data[count_data["ds"] > cutoff].copy()

evaluation_fcst = MLForecast(
    models=[LinearRegression()], freq="D", lags=[1, 7, 14], date_features=["dayofweek"]
)
evaluation_model = TwoStageForecasterWrapper(evaluation_fcst).fit(
    evaluation_train, h=7, n_windows=4
)
evaluation_forecast = evaluation_model.predict_distribution(h=evaluation_horizon)
evaluation_predictions = evaluation_forecast.ppf([0.05, 0.50, 0.95]).rename(
    columns={"lambda_t-q-5": "q_5", "lambda_t-q-50": "q_50", "lambda_t-q-95": "q_95"}
)

evaluation_results = evaluation_predictions.merge(
    evaluation_test[["unique_id", "ds", "y"]],
    on=["unique_id", "ds"],
    how="inner",
)
evaluation_results.head()

,unique_id,ds,lambda_t,r_dispersion,q_5,q_50,q_95,y
0,store_A,2024-06-15,4.399916,7.848881,1,4,9,5
1,store_A,2024-06-16,3.993489,7.848881,1,4,9,11
2,store_A,2024-06-17,3.488161,7.848881,0,3,8,3
3,store_A,2024-06-18,3.718006,7.848881,1,3,8,1
4,store_A,2024-06-19,3.563385,7.848881,0,3,8,1


### Conditional-mean evaluation and calibration table

`FirstStageForecasterEvaluator.evaluate` summarizes point-forecast behavior. The calibration table groups similar predicted means and compares their average prediction with the average observed outcome. Well-calibrated bins have a `Mean_Residual` close to zero.

In [15]:
mean_metrics = FirstStageForecasterEvaluator.evaluate(evaluation_results)
mean_calibration = FirstStageForecasterEvaluator.calibration_table(
    evaluation_results, n_bins=5
)

display(mean_metrics)
display(mean_calibration)

,wape,pbias,score,forecast_instability,false_demand_on_zero_days_avg_pred,peak_demand_deviation
0,0.5385,-0.1344,0.6729,0.0907,0.0,-0.1344


,calibration_bin,count,mean_prediction,mean_observed,mean_residual
0,"(3.287, 3.519]",6,3.408289,5.500000,2.091711
1,"(3.519, 3.887]",5,3.697237,3.800000,0.102763
2,"(3.887, 4.001]",6,3.937211,4.833333,0.896122
3,"(4.001, 4.256]",5,4.156087,3.000000,-1.156087
4,"(4.256, 4.672]",6,4.432598,5.166667,0.734069


### Probabilistic evaluation

`TwoStageForecasterEvaluator.evaluate` reports pinball loss and empirical coverage for every requested quantile available in the forecast frame. A coverage gap close to zero indicates calibrated quantiles.

In [16]:
probabilistic_metrics = TwoStageForecasterEvaluator.evaluate(
    evaluation_results, quantiles=(0.05, 0.50, 0.95)
)
probabilistic_metrics

,level,coverage_rate,interval_width_mean,mwis
0,0.9,0.8571,7.6429,16.2143


## Save and restore with joblib

Persist the fitted wrapper—not only the underlying regressor—so the calibrated family and per-series dispersion parameters are restored together. Only load files from trusted sources because `joblib.load` can execute arbitrary code during deserialization.

In [17]:
import joblib

model_path = "two_stage_forecaster.joblib"
joblib.dump(count_model, model_path)
restored_model = joblib.load(model_path)
restored_forecast = restored_model.predict_distribution(h=7)

## Family comparison

| Target | Family | CDF | PPF | PMF | `NewsvendorOptimizer.optimize` output |
|---|---|---:|---:|---:|---|
| Non-negative integer counts | Negative Binomial (default) | Yes | Yes | Yes | Integer |
| Strictly positive continuous values | `GammaFamily()` | Yes | Yes | No | Continuous |

Always match the family to the target support: Negative Binomial rejects negative or non-integer targets, while Gamma rejects zero and negative targets.